In [81]:
import os, random, re, warnings

import cv2
import joblib
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore")

imgFormats = [".bmp", ".pbm", ".pgm", ".ppm", ".sr", ".ras", ".jpeg", ".jpg", ".jpe", ".jp2", ".tiff", ".tif", ".png"]
imgs = os.listdir(f'{os.path.expanduser("~")}/Downloads/SAMPLE FILES')
imgs = [i for i in imgs if os.path.splitext(i)[1] in imgFormats and not re.match(r"[\\\/:\*\?\"<>\|]+", i)]
chosenImg = random.choice(imgs)

RESOURCE_PATH = 'PKL'
IMAGE = cv2.imdecode(np.fromfile(f"{os.path.expanduser("~")}/Downloads/SAMPLE FILES/{chosenImg}", dtype=np.uint8), cv2.IMREAD_UNCHANGED)
RATING_MAP = pd.read_csv(f"UI/assets/map/rating-map.csv")

print(f"Chosen Image: {chosenImg}")

Chosen Image: MCPC Commands.png


In [82]:
RATING_MAP

,PC1,PC2,PC3,imdb_id,cluster,r_mean,g_mean,b_mean,w_mean,rating,year
0,2.009248,0.178278,0.428975,tt3982614,1,178.300449,169.008902,165.691008,43.699666,7.1,2014
1,-1.279534,0.064459,0.713079,tt6576904,0,83.023770,72.490434,69.461921,37.773220,7.6,2017
2,-0.107444,-0.099377,1.259196,tt12777936,0,47.723639,108.482648,142.461137,46.568280,8.2,2020
3,6.024492,0.321730,0.411078,tt6131822,1,252.132573,248.907047,241.018867,80.283668,6.8,2013
4,0.321600,-0.781690,0.052011,tt0233026,1,162.741443,108.929236,78.232130,44.300936,7.7,2000
...,...,...,...,...,...,...,...,...,...,...,...
49986,-0.443277,0.142011,0.282744,tt3758246,0,123.103648,96.314998,72.453922,40.592443,7.1,2013
49987,-0.194588,-0.100362,1.363522,tt21652514,0,118.992719,106.780121,95.266741,40.203670,8.3,2022
49988,-1.645599,0.050881,-0.736517,tt0185351,0,69.606691,58.213156,63.600872,36.075614,6.3,2001
49989,-2.052146,0.489379,0.229220,tt7177906,0,86.935985,53.245589,17.351310,35.984884,6.8,2017


In [83]:
raterModel = joblib.load(f"{RESOURCE_PATH}/best_cluster_model.pkl")
pcaModel = joblib.load(f"{RESOURCE_PATH}/pca_model.pkl")
scaler = joblib.load(f"{RESOURCE_PATH}/scaler.pkl")
chosenImg

'MCPC Commands.png'

In [84]:
# Load the image
image = cv2.imread(chosenImg, IMAGE)
if image is None:
    raise FileNotFoundError(f"Failed to load the image: {IMAGE}")

# Resize the image
image = cv2.resize(image, (224, 336))

# Convert to RGB
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Calculate RGBW values
features = {}
r, g, b = cv2.split(image)
w = (r + g + b) / 3

# imputedRating = random.randint(0, 10)
imputedRating = 5
imputedYear = int(np.mean(RATING_MAP["year"]))

print(f"{imputedRating} | {imputedYear}")

features = {
    "r_mean": r.mean(),
    "g_mean": g.mean(),
    "b_mean": b.mean(),
    "w_mean": w.mean(),
    "rating": imputedRating,
    "year": imputedYear
}

5 | 2007


In [85]:
scaler.feature_names_in_

array(['r_mean', 'g_mean', 'b_mean', 'w_mean', 'rating', 'year'],
      dtype=object)

In [86]:
pcaModel.get_feature_names_out()

array(['pca0', 'pca1', 'pca2', 'pca3', 'pca4', 'pca5'], dtype=object)

In [87]:
raterModel.feature_names_in_

array(['PC1', 'PC2', 'PC3'], dtype=object)

In [88]:
df = pd.DataFrame(features, index=[0])
df.head()

,r_mean,g_mean,b_mean,w_mean,rating,year
0,48.632427,41.167955,33.736806,30.98065,5,2007


In [89]:
scaledData = scaler.transform(df)
scaledData

array([[-1.42610626, -1.42624297, -1.44637422, -0.89194896, -1.51215475,
        -0.06314353]])

In [90]:
pcaData = pcaModel.transform(scaledData)
pcaData

array([[-2.55487012,  1.0269631 , -1.22356138,  0.3407756 , -0.18669201,
         0.12656861]])

In [91]:
pcaData = pcaData[:, :raterModel.n_features_in_]
pcaData

array([[-2.55487012,  1.0269631 , -1.22356138]])

In [92]:
rating = raterModel.predict(pcaData)

print(f"Imputations: (Rating: {imputedRating}, Year: {imputedYear})")
print(f"Chosen Image: {chosenImg}")
print(f"Cluster: {rating}")
print(f"\nFeatures: {df}")

Imputations: (Rating: 5, Year: 2007)
Chosen Image: MCPC Commands.png
Cluster: [0]

Features:       r_mean     g_mean     b_mean    w_mean  rating  year
0  48.632427  41.167955  33.736806  30.98065       5  2007


In [93]:
RATING_MAP.head()

,PC1,PC2,PC3,imdb_id,cluster,r_mean,g_mean,b_mean,w_mean,rating,year
0,2.009248,0.178278,0.428975,tt3982614,1,178.300449,169.008902,165.691008,43.699666,7.1,2014
1,-1.279534,0.064459,0.713079,tt6576904,0,83.023770,72.490434,69.461921,37.773220,7.6,2017
2,-0.107444,-0.099377,1.259196,tt12777936,0,47.723639,108.482648,142.461137,46.568280,8.2,2020
3,6.024492,0.321730,0.411078,tt6131822,1,252.132573,248.907047,241.018867,80.283668,6.8,2013
4,0.321600,-0.781690,0.052011,tt0233026,1,162.741443,108.929236,78.232130,44.300936,7.7,2000


In [94]:
cluster = RATING_MAP.groupby('cluster')['rating'].mean().to_dict()
cluster

{0: 7.003458669861267, 1: 6.897072240192419}

In [95]:
print(f"{len(pcaData)} : {len(raterModel.cluster_centers_)}")

1 : 2


In [96]:
print(f"PCA features shape: {pcaData.shape}")
print(f"Cluster centers shape: {raterModel.cluster_centers_.shape}")

PCA features shape: (1, 3)
Cluster centers shape: (2, 3)


In [97]:
n_components = raterModel.cluster_centers_.shape[1]
if pcaData.shape[1] > n_components:
    pcaData = pcaData[:, :n_components]
pcaData

array([[-2.55487012,  1.0269631 , -1.22356138]])

In [98]:
distances = cdist(pcaData, raterModel.cluster_centers_)
distances

array([[1.99018712, 4.33500083]])

In [99]:
weights = 1 / (distances + 1e-10)
weights = weights / weights.sum()
weights

array([[0.68535526, 0.31464474]])

In [100]:
predicted = 0
for clusterId, weight in enumerate(weights[0]):
    predicted += cluster[clusterId] * weight
print(f"Predicted Rating for '{chosenImg}' @ cluster [{rating[0]}]: {"{:.1f}".format(predicted)} | {predicted}")

Predicted Rating for 'MCPC Commands.png' @ cluster [0]: 7.0 | 6.9699847392807985
